In [ ]:
import os
import re
import glob
import random
import warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier, early_stopping

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

# =====================================================================
# 0. CONFIGURATION & REPRODUCIBILITY
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V7 Fused (V1 + V3) Two-Pass Impedance-Calibrated Pipeline on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
SIGNAL_COLS = EMG_COLS + IMU_COLS

AUX_REG_COLS = [
    'roll', 'pitch', 'yaw', 'index_curl', 'middle_curl',
    'ring_curl', 'pinky_curl', 'pinch', 'openness', 'thumb_up'
]

GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

SEQ_LEN = 192
WINDOW_SEC = 1.4
TRAIN_STRIDE_SEC = 0.40
BATCH_SIZE = 64
EPOCHS = 35
LR = 1.1e-3
N_FOLDS = 5

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return fpath, df

# =====================================================================
# 1. TWO-PASS PER-ELECTRODE IMPEDANCE CALIBRATION + V1/V3 FEATURE ENGINE
# =====================================================================
def detrend_window_raw(df_win):
    """
    Cleanly extracts the 14 sensor channels, mean-centers per 1.4s window,
    and removes linear baseline drift (ZERO filtfilt ringing, ZERO timestamp warping!).
    """
    raw = df_win[SIGNAL_COLS].astype(np.float32).values  # (T, 14)
    raw = pd.DataFrame(raw).interpolate(method='linear', limit_direction='both').fillna(0.0).values
    T = raw.shape[0]

    centered = raw - np.mean(raw, axis=0, keepdims=True)
    if T > 4:
        t_axis = np.linspace(-1.0, 1.0, T, dtype=np.float32)[:, None]
        slope = np.sum(centered * t_axis, axis=0, keepdims=True) / (np.sum(t_axis ** 2) + 1e-8)
        detrended = centered - slope * t_axis
    else:
        detrended = centered

    # Remove slow local baseline wander on EMG using a clean 21-sample moving average (no IIR ringing)
    emg_raw = detrended[:, :8]
    k_wander = min(21, max(5, T // 10))
    emg_slow_drift = pd.DataFrame(emg_raw).rolling(window=k_wander, min_periods=1, center=True).mean().values
    emg_clean = emg_raw - emg_slow_drift

    # Smooth IMU step-holds (where imu_new == 0 repeats for 4 rows) using a 5-sample moving average
    imu_clean = pd.DataFrame(detrended[:, 8:]).rolling(window=5, min_periods=1, center=True).mean().values
    imu_clean = imu_clean - np.mean(imu_clean, axis=0, keepdims=True)

    return np.concatenate([emg_clean, imu_clean], axis=1).astype(np.float32)  # (T, 14)

def compute_session_channel_gains(window_dfs):
    """
    PASS 1: Estimates each individual electrode's contact impedance gain g_{u, c}
    and IMU scale across user u's session file so all 8 electrodes are calibrated
    to the exact same physical scale BEFORE computing spatial ratios or covariance!
    """
    rms_list = []
    for df_w in window_dfs:
        sig = detrend_window_raw(df_w)
        w_rms = np.sqrt(np.mean(sig ** 2, axis=0)) + 1e-5
        rms_list.append(w_rms)
    rms_mat = np.stack(rms_list)  # (N_win_in_session, 14)
    # Robust session scale per channel: 60th percentile of window RMSs in this session
    session_ch_gain = np.percentile(rms_mat, 60, axis=0) + 1e-4
    return session_ch_gain.astype(np.float32)  # (14,)

def extract_v7_window_features(df_win, session_ch_gain, seq_len=SEQ_LEN):
    """
    PASS 2: Combines:
      - Stream A (Exact V1 0.53790 36-channel Z-scored waveform tensor)
      - Stream B (Impedance-Calibrated V3 24-channel Spatial/Amplitude tensor)
      - Tabular Engine (Impedance-Calibrated Spatial Ratios + Riemannian Covariance + IMU Kinematics)
    """
    sig_uncal = detrend_window_raw(df_win)  # (T, 14)
    T = sig_uncal.shape[0]

    # Calibrate each channel by its per-session contact impedance gain g_{u, c}!
    sig_cal = sig_uncal / session_ch_gain[None, :]  # (T, 14)

    emg_cal = sig_cal[:, :8]
    imu_cal = sig_cal[:, 8:]

    # --- STREAM A: V1's Exact 36-Channel Per-Window Z-Scored Waveform (Proven 0.53790) ---
    delta_uncal = np.diff(sig_uncal, axis=0, prepend=sig_uncal[:1, :])
    emg_u = sig_uncal[:, :8]
    emg_p = np.roll(emg_u, 1, axis=0); emg_p[0] = emg_u[0]
    emg_n = np.roll(emg_u, -1, axis=0); emg_n[-1] = emg_u[-1]
    tkeo_u = (emg_u ** 2) - (emg_p * emg_n)

    v1_raw_stack = np.concatenate([sig_uncal, delta_uncal, tkeo_u], axis=1)  # (T, 36)
    old_idx = np.linspace(0.0, 1.0, T)
    new_idx = np.linspace(0.0, 1.0, seq_len)

    v1_res = np.zeros((seq_len, 36), dtype=np.float32)
    for c in range(36):
        v1_res[:, c] = np.interp(new_idx, old_idx, v1_raw_stack[:, c])
    v1_norm = np.clip((v1_res - np.mean(v1_res, axis=0, keepdims=True)) / (np.std(v1_res, axis=0, keepdims=True) + 1e-5), -8.0, 8.0)

    # --- STREAM B: Impedance-Calibrated Spatial & Envelope Stream (24 Channels) ---
    env_k = max(5, T // 14)
    emg_env_cal = np.sqrt(pd.DataFrame(emg_cal ** 2).rolling(window=env_k, min_periods=1, center=True).mean().values + 1e-6)
    spatial_prop_cal = emg_env_cal / (np.sum(emg_env_cal, axis=1, keepdims=True) + 1e-4)

    emg_cal_comp = np.sign(emg_cal) * np.log1p(np.abs(emg_cal))
    env_cal_comp = np.log1p(emg_env_cal)
    imu_cal_comp = np.sign(imu_cal) * np.log1p(np.abs(imu_cal))

    # 8 (emg_cal_comp) + 8 (env_cal_comp) + 8 (spatial_prop_cal) + 6 (imu_cal_comp) = 30 channels
    v3_raw_stack = np.concatenate([emg_cal_comp, env_cal_comp, spatial_prop_cal, imu_cal_comp], axis=1)  # (T, 30)
    v3_res = np.zeros((seq_len, 30), dtype=np.float32)
    for c in range(30):
        v3_res[:, c] = np.interp(new_idx, old_idx, v3_raw_stack[:, c])

    # Full 66-Channel Sequence Tensor: [36 V1 Shape Channels + 30 Calibrated Spatial/Amplitude Channels]
    seq_tensor = np.concatenate([v1_norm, np.clip(v3_res, -8.0, 8.0)], axis=1).T.astype(np.float32)  # (66, 192)

    # --- PART C: IMPEDANCE-CALIBRATED BIOMECHANICAL TABULAR FEATURES ---
    feats = []
    emg_diff_cal = np.diff(emg_cal, axis=0, prepend=emg_cal[:1, :])

    # 1. Calibrated Per-Channel Amplitudes (Now 100% free of electrode contact impedance bias!)
    rms_cal = np.sqrt(np.mean(emg_cal ** 2, axis=0)) + 1e-5
    mav_cal = np.mean(np.abs(emg_cal), axis=0) + 1e-5
    wl_cal = np.mean(np.abs(emg_diff_cal), axis=0) + 1e-5
    p90_cal = np.percentile(np.abs(emg_cal), 90, axis=0) + 1e-5
    p50_cal = np.percentile(np.abs(emg_cal), 50, axis=0) + 1e-5
    zc_cal = np.sum(np.diff(np.sign(emg_cal), axis=0) != 0, axis=0) / float(T)

    feats.extend(np.log1p(rms_cal))
    feats.extend(np.log1p(mav_cal))
    feats.extend(np.log1p(wl_cal))
    feats.extend(np.log1p(p90_cal))
    feats.extend(np.log1p(p50_cal))
    feats.extend(zc_cal)

    # 2. Calibrated Spatial Proportions Across the 8 Forearm Electrodes (32 features)
    rms_prop = rms_cal / np.sum(rms_cal)
    wl_prop = wl_cal / np.sum(wl_cal)
    p90_prop = p90_cal / np.sum(p90_cal)
    mav_prop = mav_cal / np.sum(mav_cal)
    feats.extend(rms_prop)
    feats.extend(wl_prop)
    feats.extend(p90_prop)
    feats.extend(mav_prop)

    # Also include uncalibrated spatial ratio so trees can compare both
    rms_uncal = np.sqrt(np.mean(emg_u ** 2, axis=0)) + 1e-5
    feats.extend(rms_uncal / np.sum(rms_uncal))

    # Global Calibrated Muscle Energy & Spatial Concentration (10 features)
    tot_rms_cal = float(np.log1p(np.sum(rms_cal)))
    tot_wl_cal = float(np.log1p(np.sum(wl_cal)))
    entropy = float(-np.sum(rms_prop * np.log(rms_prop + 1e-6)))
    feats.extend([
        tot_rms_cal, tot_wl_cal, entropy,
        float(np.max(rms_prop)), float(np.min(rms_prop)), float(np.std(rms_prop)),
        float(np.log1p(np.max(rms_cal))), float(np.log1p(np.median(rms_cal))),
        float(np.max(rms_prop) / (np.median(rms_prop) + 1e-4)),
        float(np.argmax(rms_prop))
    ])

    # 3. Circular Forearm Ring Antagonist Muscle Contrasts (24 features)
    for i in range(8):
        feats.append(float(rms_prop[i] - rms_prop[(i + 4) % 8]))  # Opposing flexor vs extensor
        feats.append(float(rms_prop[i] - rms_prop[(i + 1) % 8]))  # Adjacent neighbor
        feats.append(float( np.log1p(rms_cal[i]) - np.log1p(rms_cal[(i + 4) % 8]) ))

    # 4. Calibrated Riemannian Log-Euclidean Covariance & Correlation Matrices (72 features)
    # Covariance on impedance-calibrated EMG (captures BOTH calibrated power AND cross-channel synergy!)
    cov_cal = (emg_cal.T @ emg_cal) / float(T) + 1e-3 * np.eye(8)
    eigvals_c, eigvecs_c = np.linalg.eigh(cov_cal)
    eigvals_c = np.clip(eigvals_c, 1e-4, None)
    log_cov_cal = eigvecs_c @ np.diag(np.log(eigvals_c)) @ eigvecs_c.T
    feats.extend(log_cov_cal[np.triu_indices(8, k=0)])  # 36 features
    feats.extend(np.log(eigvals_c))                     # 8 features

    # Pure correlation matrix (scale-free co-contraction between electrode pairs)
    corr_mat = np.nan_to_num(np.corrcoef(emg_cal.T), nan=0.0)
    feats.extend(corr_mat[np.triu_indices(8, k=1)])     # 28 features

    # 5. Sub-Window Temporal Evolution: Onset vs Hold vs Release (36 features)
    for sub in np.array_split(emg_env_cal, 3, axis=0):
        s_m = np.mean(sub, axis=0) + 1e-5
        feats.extend(np.log1p(s_m))
        feats.extend(s_m / np.sum(s_m))
    feats.extend(np.percentile(emg_env_cal, 90, axis=0) - np.percentile(emg_env_cal, 20, axis=0))
    feats.extend(np.mean(emg_env_cal[T // 4 : 3 * T // 4], axis=0) / (np.mean(emg_env_cal[:T // 4], axis=0) + 1e-3))

    # 6. FFT Frequency Band Ratios (24 features)
    fft_p = np.abs(np.fft.rfft(emg_cal, axis=0)) ** 2
    if len(fft_p) > 6:
        b1, b2, b3 = np.array_split(fft_p[1:], 3, axis=0)
        p1, p2, p3 = np.mean(b1, axis=0) + 1e-5, np.mean(b2, axis=0) + 1e-5, np.mean(b3, axis=0) + 1e-5
        ptot = p1 + p2 + p3
        feats.extend(p1 / ptot)
        feats.extend(p2 / ptot)
        feats.extend(p3 / ptot)
    else:
        feats.extend(np.zeros(24))

    # 7. Calibrated & Raw IMU 3D Kinematics (55 features)
    imu_rms_cal = np.sqrt(np.mean(imu_cal ** 2, axis=0)) + 1e-5
    imu_p90_cal = np.percentile(np.abs(imu_cal), 90, axis=0)
    imu_min_cal = np.percentile(imu_cal, 10, axis=0)
    imu_max_cal = np.percentile(imu_cal, 90, axis=0)

    acc_prop = imu_rms_cal[:3] / (np.sum(imu_rms_cal[:3]) + 1e-5)
    gyro_prop = imu_rms_cal[3:] / (np.sum(imu_rms_cal[3:]) + 1e-5)

    feats.extend(np.log1p(imu_rms_cal))
    feats.extend(np.log1p(imu_p90_cal))
    feats.extend(imu_min_cal)
    feats.extend(imu_max_cal)
    feats.extend(acc_prop)
    feats.extend(gyro_prop)

    imu_traj = np.cumsum(imu_cal, axis=0) / float(T)
    feats.extend(imu_traj[-1, :])
    feats.extend(np.mean(imu_cal[T // 2:], axis=0) - np.mean(imu_cal[:T // 2], axis=0))

    acc_mag = np.sqrt(np.sum(imu_cal[:, :3] ** 2, axis=1))
    gyro_mag = np.sqrt(np.sum(imu_cal[:, 3:] ** 2, axis=1))
    feats.extend([
        float(np.log1p(np.mean(acc_mag))), float(np.log1p(np.std(acc_mag))),
        float(np.log1p(np.mean(gyro_mag))), float(np.log1p(np.std(gyro_mag))),
        float(np.log1p(np.sum(rms_cal)) / (np.log1p(np.mean(gyro_mag)) + 0.1))
    ])

    imu_corr = np.nan_to_num(np.corrcoef(imu_cal.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return seq_tensor, tab_vec

# =====================================================================
# 2. PER-SESSION FEATURE WHITENING & RANK ALIGNMENT (FROM VERSION 3)
# =====================================================================
def append_session_aligned_features(X_tab, session_ids):
    N, D = X_tab.shape
    X_sess_z = np.zeros_like(X_tab)
    rank_cols = list(range(32))
    X_sess_rank = np.zeros((N, len(rank_cols)), dtype=np.float32)

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 3:
            continue
        sub = X_tab[mask]
        med = np.median(sub, axis=0, keepdims=True)
        iqr = (np.percentile(sub, 75, axis=0, keepdims=True) - np.percentile(sub, 25, axis=0, keepdims=True)) + 1e-4
        X_sess_z[mask] = np.clip((sub - med) / iqr, -6.0, 6.0)

        for idx_r, col_r in enumerate(rank_cols):
            vals = sub[:, col_r]
            ranks = np.argsort(np.argsort(vals)).astype(np.float32) / float(max(1, len(vals) - 1))
            X_sess_rank[mask, idx_r] = ranks - 0.5

    return np.concatenate([X_tab, X_sess_z, X_sess_rank], axis=1)

# =====================================================================
# 3. TWO-PASS PARALLEL DATASET BUILDER
# =====================================================================
def process_single_train_session(args):
    session_idx, (fpath, df) = args
    if 'label_source' in df.columns:
        df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)

    label_col = 'label' if 'label' in df.columns else 'gesture'
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
    if len(df) == 0:
        return None

    time_jump = df['t'].diff().abs() > 0.25
    label_change = df[label_col] != df[label_col].shift()
    df['seg_id'] = (time_jump | label_change).cumsum()

    raw_windows, cls_list, reg_list, mask_list, anchor_list = [], [], [], [], []

    for _, seg in df.groupby('seg_id'):
        t_start, t_end = seg['t'].iloc[0], seg['t'].iloc[-1]
        gesture_id = GESTURE_MAP[seg[label_col].iloc[0]]

        curr_t = t_start
        step_idx = 0
        while curr_t + WINDOW_SEC <= t_end + 0.08:
            win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
            if len(win) >= 35:
                r_t = np.zeros(len(AUX_REG_COLS), dtype=np.float32)
                r_m = np.zeros(len(AUX_REG_COLS), dtype=np.float32)
                for idx, col in enumerate(AUX_REG_COLS):
                    if col in win.columns:
                        vals = pd.to_numeric(win[col], errors='coerce').dropna().values
                        if len(vals) > 0:
                            r_t[idx] = float(np.mean(vals))
                            r_m[idx] = 1.0

                raw_windows.append(win)
                cls_list.append(gesture_id)
                reg_list.append(r_t)
                mask_list.append(r_m)
                anchor_list.append(1 if (step_idx % 3 == 0) else 0)
                step_idx += 1
            curr_t += TRAIN_STRIDE_SEC

    if not raw_windows:
        return None

    # PASS 1: Compute per-electrode contact impedance gains from this session's non-overlapping anchor windows
    anchor_wins = [w for w, is_a in zip(raw_windows, anchor_list) if is_a == 1]
    session_ch_gain = compute_session_channel_gains(anchor_wins if len(anchor_wins) >= 5 else raw_windows)

    # PASS 2: Extract Impedance-Calibrated Waveforms & Tabular Features
    seq_list, tab_list = [], []
    for win in raw_windows:
        s_t, t_v = extract_v7_window_features(win, session_ch_gain, seq_len=SEQ_LEN)
        seq_list.append(s_t)
        tab_list.append(t_v)

    n = len(seq_list)
    return (
        np.stack(seq_list), np.stack(tab_list), np.array(cls_list, dtype=np.int64),
        np.stack(reg_list), np.stack(mask_list),
        np.full(n, session_idx, dtype=np.int64), np.array(anchor_list, dtype=np.int64)
    )

def load_v7_datasets(data_dir):
    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Parallel-loading {len(train_files)} train sessions and {len(test_files)} test sessions...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        train_loaded = list(ex.map(read_aura_excel_fast, train_files))
        test_loaded = list(ex.map(read_aura_excel_fast, test_files))
        train_res = list(ex.map(process_single_train_session, enumerate(train_loaded)))

    train_res = [r for r in train_res if r is not None]
    X_seq = np.concatenate([r[0] for r in train_res], axis=0)
    X_tab = np.concatenate([r[1] for r in train_res], axis=0)
    y_cls = np.concatenate([r[2] for r in train_res], axis=0)
    y_reg = np.concatenate([r[3] for r in train_res], axis=0)
    reg_mask = np.concatenate([r[4] for r in train_res], axis=0)
    groups = np.concatenate([r[5] for r in train_res], axis=0)
    is_anchor = np.concatenate([r[6] for r in train_res], axis=0)

    for j in range(y_reg.shape[1]):
        valid = reg_mask[:, j] > 0.5
        if np.sum(valid) > 10:
            y_reg[valid, j] = np.clip(
                (y_reg[valid, j] - np.mean(y_reg[valid, j])) / (np.std(y_reg[valid, j]) + 1e-5),
                -4.0, 4.0
            )

    # Two-Pass Processing on Each Test Session (test_1.xlsx .. test_5.xlsx)
    X_test_seq, X_test_tab, test_ids, test_sessions = [], [], [], []
    for test_sess_idx, (fpath, df) in enumerate(test_loaded):
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        sess_wins, sess_ids = [], []
        for act_id, win in df.groupby(id_col, sort=False):
            sess_wins.append(win)
            sess_ids.append(act_id)

        # PASS 1: Estimate test user's per-electrode contact impedance gain
        test_ch_gain = compute_session_channel_gains(sess_wins)

        # PASS 2: Extract calibrated features for each test action
        for act_id, win in zip(sess_ids, sess_wins):
            s_t, t_v = extract_v7_window_features(win, test_ch_gain, seq_len=SEQ_LEN)
            X_test_seq.append(s_t)
            X_test_tab.append(t_v)
            test_ids.append(act_id)
            test_sessions.append(test_sess_idx)

    X_test_seq = np.stack(X_test_seq)
    X_test_tab = np.stack(X_test_tab)
    test_sessions = np.array(test_sessions, dtype=np.int64)

    # Append Within-Session Whitened & Rank Features
    X_tab = append_session_aligned_features(X_tab, groups)
    X_test_tab = append_session_aligned_features(X_test_tab, test_sessions)

    print(f"Train Windows: {len(X_seq)} (Clean Anchors: {np.sum(is_anchor)}) | Seq: {X_seq.shape} | Tabular: {X_tab.shape}")
    print(f"Test Actions:  {len(X_test_seq)} across {len(np.unique(test_sessions))} test sessions")
    return X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor, X_test_seq, X_test_tab, test_ids, test_sessions

# =====================================================================
# 4. FUSED V1 + V3 DEEP ARCHITECTURES & TABULAR RESIDUAL MLP
# =====================================================================
class AuraV7Dataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, y_reg=None, reg_mask=None, is_train=True):
        self.X_seq = X_seq
        self.X_tab = X_tab
        self.y_cls = y_cls
        self.y_reg = y_reg
        self.reg_mask = reg_mask
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        x = self.X_seq[idx].copy()
        tab = self.X_tab[idx].copy()

        if self.is_train:
            if random.random() < 0.55:
                shift = random.randint(-8, 8)
                x = np.roll(x, shift=shift, axis=-1)
            if random.random() < 0.45:
                scale = np.random.uniform(0.90, 1.10, size=(x.shape[0], 1)).astype(np.float32)
                noise = np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
                x = x * scale + noise
            if random.random() < 0.25:
                t0 = random.randint(0, x.shape[1] - 16)
                x[:, t0 : t0 + 16] = 0.0

        if self.y_cls is None:
            return torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.tensor(self.y_reg[idx], dtype=torch.float32),
            torch.tensor(self.reg_mask[idx], dtype=torch.float32)
        )

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.GELU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.fc(x.mean(dim=-1)).unsqueeze(-1)
        return x * w

class AuraHybridNetV1Upgraded(nn.Module):
    """
    Upgraded Version 1 (0.53790) Backbone:
      - Applies InstanceNorm1d ONLY to Stream A (first 36 V1 shape channels)
      - Applies BatchNorm1d to Stream B (30 Impedance-Calibrated Spatial/Amplitude channels)
      - Fuses Multi-Scale Conv1D + SEBlock1D + Transformer + Attention Pooling + Tabular MLP!
    """
    def __init__(self, in_channels=66, tab_dim=610, d_model=128, num_classes=6, num_reg=10):
        super().__init__()
        self.inst_norm_v1 = nn.InstanceNorm1d(36, affine=True)
        self.bn_v3 = nn.BatchNorm1d(in_channels - 36)

        self.conv_k3 = nn.Conv1d(in_channels, d_model // 2, kernel_size=3, padding=1)
        self.conv_k7 = nn.Conv1d(in_channels, d_model // 4, kernel_size=7, padding=3)
        self.conv_k15 = nn.Conv1d(in_channels, d_model // 4, kernel_size=15, padding=7)

        self.cnn_encoder = nn.Sequential(
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            SEBlock1D(d_model),
            nn.Dropout(0.15),
            nn.MaxPool1d(2, 2),  # 192 -> 96
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            SEBlock1D(d_model),
            nn.MaxPool1d(2, 2)   # 96 -> 48
        )

        self.pos_emb = nn.Parameter(torch.randn(1, 48, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=d_model * 2,
            dropout=0.20, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.attn_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim),
            nn.Dropout(0.25),
            nn.Linear(tab_dim, 160),
            nn.BatchNorm1d(160),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(160, 96),
            nn.BatchNorm1d(96),
            nn.GELU()
        )

        fused_dim = d_model * 2 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(fused_dim + num_reg),
            nn.Dropout(0.30),
            nn.Linear(fused_dim + num_reg, 128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_seq, x_tab):
        x_v1 = self.inst_norm_v1(x_seq[:, :36, :])
        x_v3 = self.bn_v3(x_seq[:, 36:, :])
        x = torch.cat([x_v1, x_v3], dim=1)

        x = torch.cat([self.conv_k3(x), self.conv_k7(x), self.conv_k15(x)], dim=1)
        x = self.cnn_encoder(x)
        seq = self.transformer(x.permute(0, 2, 1) + self.pos_emb)

        w = F.softmax(self.attn_pool(seq), dim=1)
        attn_p = torch.sum(seq * w, dim=1)
        max_p = torch.max(seq, dim=1)[0]
        tab_e = self.tab_mlp(x_tab)

        fused = torch.cat([attn_p, max_p, tab_e], dim=-1)
        reg_p = self.reg_head(fused)
        logits = self.cls_head(torch.cat([fused, reg_p], dim=-1))
        return logits, reg_p

class InceptionWaveBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        mid = out_c // 4
        self.b1 = nn.Conv1d(in_c, mid, kernel_size=1)
        self.b2 = nn.Conv1d(in_c, mid, kernel_size=5, padding=2)
        self.b3 = nn.Conv1d(in_c, mid, kernel_size=11, padding=5)
        self.b4 = nn.Conv1d(in_c, mid, kernel_size=23, padding=11)
        self.bn = nn.BatchNorm1d(out_c)
        self.short = nn.Conv1d(in_c, out_c, kernel_size=1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        out = torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)
        return F.gelu(self.bn(out) + self.short(x))

class InceptionBiGRUNetV3(nn.Module):
    """
    Upgraded Version 3 (0.54557) InceptionWaveNet + BiGRU + Tabular Fusion Backbone.
    """
    def __init__(self, in_channels=66, tab_dim=610, num_classes=6, num_reg=10):
        super().__init__()
        self.inc1 = InceptionWaveBlock(in_channels, 96)
        self.pool1 = nn.AvgPool1d(2, 2)
        self.inc2 = InceptionWaveBlock(96, 128)
        self.pool2 = nn.AvgPool1d(2, 2)
        self.inc3 = InceptionWaveBlock(128, 128)
        self.bigru = nn.GRU(128, 64, num_layers=1, batch_first=True, bidirectional=True)

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim),
            nn.Dropout(0.25),
            nn.Linear(tab_dim, 160),
            nn.BatchNorm1d(160),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(160, 96),
            nn.GELU()
        )

        # Mean + Std + Max pooling (128 * 3 = 384) + 96 = 480
        fused_dim = 128 * 3 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(fused_dim + num_reg),
            nn.Dropout(0.30),
            nn.Linear(fused_dim + num_reg, 128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_seq, x_tab):
        h = self.pool1(self.inc1(x_seq))
        h = self.pool2(self.inc2(h))
        h = self.inc3(h)
        seq_g, _ = self.bigru(h.permute(0, 2, 1))
        h = h + seq_g.permute(0, 2, 1)

        mean_p = h.mean(dim=-1)
        std_p = h.std(dim=-1)
        max_p = h.max(dim=-1)[0]
        tab_e = self.tab_mlp(x_tab)

        fused = torch.cat([mean_p, std_p, max_p, tab_e], dim=-1)
        reg_p = self.reg_head(fused)
        logits = self.cls_head(torch.cat([fused, reg_p], dim=-1))
        return logits, reg_p

class AuraCompositeLoss(nn.Module):
    def __init__(self, num_classes=6, smoothing=0.06, gamma=2.0, f1_w=0.35, reg_w=0.25):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.gamma = gamma
        self.f1_w = f1_w
        self.reg_w = reg_w

    def forward(self, logits, y_cls, reg_p, y_reg, reg_m):
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)

        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, y_cls.unsqueeze(1), 1.0 - self.smoothing)

        focal = torch.mean(torch.sum(-true_dist * ((1.0 - probs) ** self.gamma) * log_probs, dim=-1))

        one_hot = F.one_hot(y_cls, num_classes=self.num_classes).float()
        tp = torch.sum(probs * one_hot, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot), dim=0)
        fn = torch.sum((1.0 - probs) * one_hot, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)

        loss = (1.0 - self.f1_w) * focal + self.f1_w * (1.0 - torch.mean(soft_f1))

        valid = torch.sum(reg_m)
        if valid > 0:
            reg_l1 = F.smooth_l1_loss(reg_p, y_reg, reduction='none')
            loss = loss + self.reg_w * (torch.sum(reg_l1 * reg_m) / (valid + 1e-5))
        return loss

def predict_with_tta(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for batch in loader:
            bx_s, bx_t = batch[0].to(device), batch[1].to(device)
            l0, _ = model(bx_s, bx_t)
            l1, _ = model(torch.roll(bx_s, shifts=-5, dims=-1), bx_t)
            l2, _ = model(torch.roll(bx_s, shifts=5, dims=-1), bx_t)
            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            probs_list.append(p.cpu().numpy())
    return np.concatenate(probs_list, axis=0)

# =====================================================================
# 5. LOGIT-SPACE SESSION SELF-CALIBRATION (SOLVES HIGH-DIM CURSE)
# =====================================================================
def calibrate_session_logits(probs, session_ids, power=0.45, n_iter=3):
    """
    Performs low-dimensional (6-D probability simplex) Sinkhorn + Prototype Calibration
    within each session. Unlike high-dimensional feature centroids, 6-D probability
    space never suffers from the curse of dimensionality!
    """
    cal_probs = probs.copy()
    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 12:
            continue
        P = cal_probs[mask].copy() + 1e-6
        for _ in range(n_iter):
            P = P / np.sum(P, axis=1, keepdims=True)
            col_mean = np.mean(P, axis=0, keepdims=True)
            P = P * ((1.0 / 6.0) / (col_mean + 1e-6)) ** power
            P = P / np.sum(P, axis=1, keepdims=True)
        cal_probs[mask] = P
    return cal_probs

# =====================================================================
# 6. 5-FOLD EXECUTION: V1-HYBRID + V3-INCEPTION + LGBM + CATBOOST
# =====================================================================
(X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor,
 X_test_seq, X_test_tab, test_ids, test_sessions) = load_v7_datasets(DATA_DIR)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)

oof_v1 = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_v3 = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_lgb = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_cat = np.zeros((len(X_seq), 6), dtype=np.float32)

test_v1 = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_v3 = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_lgb = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_cat = np.zeros((len(X_test_seq), 6), dtype=np.float32)

criterion = AuraCompositeLoss()

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
    print(f"\n==================== FOLD {fold + 1} / {N_FOLDS} ====================")
    clean_val_idx = val_idx[is_anchor[val_idx] == 1]
    anchor_submask = (is_anchor[val_idx] == 1)

    scaler = StandardScaler()
    X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -8.0, 8.0)
    X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -8.0, 8.0)
    X_tab_val_clean = np.clip(scaler.transform(X_tab[clean_val_idx]), -8.0, 8.0)
    X_tab_tst = np.clip(scaler.transform(X_test_tab), -8.0, 8.0)

    # --- 1. Feature-Pruned LightGBM on Impedance-Calibrated & Session-Aligned Features ---
    scout = LGBMClassifier(
        n_estimators=200, learning_rate=0.04, max_depth=4, num_leaves=15,
        subsample=0.8, colsample_bytree=0.6, random_state=42 + fold, n_jobs=-1, verbose=-1
    )
    scout.fit(X_tab_trn, y_cls[trn_idx])
    keep_k = int(X_tab.shape[1] * 0.75)
    keep_cols = np.argsort(scout.feature_importances_)[::-1][:keep_k]

    lgb_model = LGBMClassifier(
        n_estimators=1500, learning_rate=0.025, max_depth=5, num_leaves=22,
        min_child_samples=28, min_split_gain=0.01, subsample=0.80, subsample_freq=1,
        colsample_bytree=0.60, reg_alpha=2.0, reg_lambda=4.0,
        random_state=100 + fold, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(
        X_tab_trn[:, keep_cols], y_cls[trn_idx],
        eval_set=[(X_tab_val_clean[:, keep_cols], y_cls[clean_val_idx])],
        callbacks=[early_stopping(90, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_tab_val[:, keep_cols])
    test_lgb += lgb_model.predict_proba(X_tab_tst[:, keep_cols]) / N_FOLDS
    lgb_f1 = f1_score(y_cls[clean_val_idx], np.argmax(oof_lgb[clean_val_idx], axis=1), average='macro')

    # --- 2. Pruned CatBoost ---
    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(
            iterations=1500, learning_rate=0.035, depth=5, l2_leaf_reg=6.0,
            random_seed=100 + fold, task_type='CPU', verbose=False
        )
        cat_model.fit(
            X_tab_trn[:, keep_cols], y_cls[trn_idx],
            eval_set=(X_tab_val_clean[:, keep_cols], y_cls[clean_val_idx]),
            early_stopping_rounds=90, verbose=False
        )
        oof_cat[val_idx] = cat_model.predict_proba(X_tab_val[:, keep_cols])
        test_cat += cat_model.predict_proba(X_tab_tst[:, keep_cols]) / N_FOLDS
        cat_f1 = f1_score(y_cls[clean_val_idx], np.argmax(oof_cat[clean_val_idx], axis=1), average='macro')
    else:
        oof_cat[val_idx] = oof_lgb[val_idx]
        test_cat += test_lgb / N_FOLDS
        cat_f1 = lgb_f1

    print(f"  [Impedance-Calibrated Trees] LGBM Val F1: {lgb_f1:.4f} | CatBoost Val F1: {cat_f1:.4f}")

    trn_ds = AuraV7Dataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], y_reg[trn_idx], reg_mask[trn_idx], is_train=True)
    val_ds = AuraV7Dataset(X_seq[val_idx], X_tab_val, is_train=False)
    tst_ds = AuraV7Dataset(X_test_seq, X_tab_tst, is_train=False)

    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False)

    # --- 3. Train Model 1: AuraHybridNetV1Upgraded (V1 0.53790 + V3 Calibrated Fusion) ---
    m1 = AuraHybridNetV1Upgraded(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    opt1 = torch.optim.AdamW(m1.parameters(), lr=LR, weight_decay=1.5e-2)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EPOCHS, eta_min=1e-5)
    best_m1_f1, best_m1_p, ckpt1 = 0.0, None, f"v7_m1_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        m1.train()
        for bx_s, bx_t, by_c, by_r, bm in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_r, bm = by_c.to(DEVICE), by_r.to(DEVICE), bm.to(DEVICE)
            opt1.zero_grad()
            logits, reg_p = m1(bx_s, bx_t)
            loss = criterion(logits, by_c, reg_p, by_r, bm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m1.parameters(), 1.0)
            opt1.step()
        sch1.step()

        vp = predict_with_tta(m1, val_loader, DEVICE)
        cf1 = f1_score(y_cls[val_idx][anchor_submask], np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > best_m1_f1:
            best_m1_f1, best_m1_p = cf1, vp
            torch.save(m1.state_dict(), ckpt1)

    oof_v1[val_idx] = best_m1_p
    m1.load_state_dict(torch.load(ckpt1, map_location=DEVICE))
    test_v1 += predict_with_tta(m1, tst_loader, DEVICE) / N_FOLDS

    # --- 4. Train Model 2: InceptionBiGRUNetV3 (Version 3 0.54557 Upgraded Backbone) ---
    m2 = InceptionBiGRUNetV3(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    opt2 = torch.optim.AdamW(m2.parameters(), lr=LR, weight_decay=1.5e-2)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS, eta_min=1e-5)
    best_m2_f1, best_m2_p, ckpt2 = 0.0, None, f"v7_m2_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        m2.train()
        for bx_s, bx_t, by_c, by_r, bm in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_r, bm = by_c.to(DEVICE), by_r.to(DEVICE), bm.to(DEVICE)
            opt2.zero_grad()
            logits, reg_p = m2(bx_s, bx_t)
            loss = criterion(logits, by_c, reg_p, by_r, bm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
            opt2.step()
        sch2.step()

        vp = predict_with_tta(m2, val_loader, DEVICE)
        cf1 = f1_score(y_cls[val_idx][anchor_submask], np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > best_m2_f1:
            best_m2_f1, best_m2_p = cf1, vp
            torch.save(m2.state_dict(), ckpt2)

    oof_v3[val_idx] = best_m2_p
    m2.load_state_dict(torch.load(ckpt2, map_location=DEVICE))
    test_v3 += predict_with_tta(m2, tst_loader, DEVICE) / N_FOLDS

    print(f"  [Deep Models] V1-Hybrid Val F1: {best_m1_f1:.4f} | V3-InceptionBiGRU Val F1: {best_m2_f1:.4f}")

# =====================================================================
# 7. STRICT OOF-GATED ENSEMBLE BLENDING & SELF-CALIBRATION
# =====================================================================
clean_mask = (is_anchor == 1)
y_clean = y_cls[clean_mask]

f1_v1 = f1_score(y_clean, np.argmax(oof_v1[clean_mask], axis=1), average='macro')
f1_v3 = f1_score(y_clean, np.argmax(oof_v3[clean_mask], axis=1), average='macro')
f1_lgb = f1_score(y_clean, np.argmax(oof_lgb[clean_mask], axis=1), average='macro')
f1_cat = f1_score(y_clean, np.argmax(oof_cat[clean_mask], axis=1), average='macro')

best_raw_f1 = 0.0
best_w = (0.30, 0.30, 0.20, 0.20)

for w1 in np.linspace(0.10, 0.50, 9):
    for w2 in np.linspace(0.10, 0.50, 9):
        for w3 in np.linspace(0.10, 0.50, 9):
            w4 = max(0.0, 1.0 - w1 - w2 - w3)
            if w1 + w2 + w3 > 1.0:
                continue
            blend = w1 * oof_v1[clean_mask] + w2 * oof_v3[clean_mask] + w3 * oof_lgb[clean_mask] + w4 * oof_cat[clean_mask]
            s = f1_score(y_clean, np.argmax(blend, axis=1), average='macro')
            if s > best_raw_f1:
                best_raw_f1 = s
                best_w = (w1, w2, w3, w4)

oof_blend = best_w[0] * oof_v1 + best_w[1] * oof_v3 + best_w[2] * oof_lgb + best_w[3] * oof_cat
test_blend = best_w[0] * test_v1 + best_w[1] * test_v3 + best_w[2] * test_lgb + best_w[3] * test_cat

# OOF-Gated Session Marginal Calibration (Applied ONLY if it strictly improves OOF Macro-F1!)
best_cal_f1 = best_raw_f1
best_power = 0.0
oof_cal = oof_blend.copy()

for pwr in [0.15, 0.25, 0.35, 0.45, 0.55]:
    cand_oof = calibrate_session_logits(oof_blend, groups, power=pwr, n_iter=3)
    s = f1_score(y_clean, np.argmax(cand_oof[clean_mask], axis=1), average='macro')
    if s > best_cal_f1:
        best_cal_f1 = s
        best_power = pwr
        oof_cal = cand_oof

if best_power > 0.0:
    test_cal = calibrate_session_logits(test_blend, test_sessions, power=best_power, n_iter=3)
else:
    test_cal = test_blend

# Conservative Class Multiplier Tuning
class_mult = np.ones(6, dtype=np.float32)
tuned_f1 = best_cal_f1
for _ in range(3):
    for c in range(6):
        for m in np.linspace(0.85, 1.20, 36):
            cand = class_mult.copy()
            cand[c] = m
            s = f1_score(y_clean, np.argmax(oof_cal[clean_mask] * cand[None, :], axis=1), average='macro')
            if s > tuned_f1:
                tuned_f1 = s
                class_mult = cand

print("\n========================================================")
print(f"Clean OOF Macro-F1 (V1-Hybrid Upgraded):     {f1_v1:.4f}")
print(f"Clean OOF Macro-F1 (V3-InceptionBiGRU):      {f1_v3:.4f}")
print(f"Clean OOF Macro-F1 (Impedance-Calib LGBM):   {f1_lgb:.4f}")
print(f"Clean OOF Macro-F1 (Impedance-Calib CatB):   {f1_cat:.4f}")
print(f"Clean OOF Macro-F1 (4-Model Ensemble Raw):   {best_raw_f1:.4f} | Weights: {np.round(best_w, 2)}")
print(f"Clean OOF Macro-F1 (+Session Calibration):   {best_cal_f1:.4f} | Power: {best_power}")
print(f"Clean OOF Macro-F1 (+Tuned Multipliers):     {tuned_f1:.4f}")
print("========================================================")
print(classification_report(y_clean, np.argmax(oof_cal[clean_mask] * class_mult[None, :], axis=1), target_names=GESTURES))

final_preds = np.argmax(test_cal * class_mult[None, :], axis=1)
final_labels = [INV_GESTURE_MAP[i] for i in final_preds]

sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        pred_map = dict(zip(test_ids, final_labels))
        sub[target_col] = sub[id_col].map(pred_map).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("\nSaved V7 submission.csv! Test Set Prediction Distribution:")
print(sub.iloc[:, 1].value_counts())